# M1 — Dataset and Preprocessing

Locating numbers in text images

Assigned member: Idhant Kadela (230481)

Prepared supporting notebook. Review and run before adopting its contribution declaration.

The calculations were executed against the supplied project files during report preparation. This portable copy starts with a Colab Drive mount. Adjust PROJECT if the folder has a different name. Install numpy, pandas, matplotlib and pillow if absent.

This notebook audits the existing prepared dataset and generates statistics and figures. The original raw-SVHN conversion script and random split manifest remain to be recovered.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import os
import zipfile
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# In Colab, mount Drive first and use the original project folder.
PROJECT = Path(os.environ.get('EE604_PROJECT', '/content/drive/MyDrive/NumericalTextLocalization'))
DATA = Path(os.environ.get('EE604_DATA', '/content/ee604_dataset'))
OUTPUT = Path(os.environ.get('EE604_OUTPUT', str(PROJECT / 'results/M1_dataset_analysis')))
OUTPUT.mkdir(parents=True, exist_ok=True)
if not (DATA / 'data.yaml').exists():
    with zipfile.ZipFile(PROJECT / 'datasets/SVHN_YOLO.zip') as archive:
        archive.extractall(DATA)

statistics = []
class_counts = {}
length_counts = {}
for split in ['train', 'val', 'test']:
    image_paths = sorted((DATA / 'images' / split).glob('*.jpg'))
    label_paths = sorted((DATA / 'labels' / split).glob('*.txt'))
    assert {p.stem for p in image_paths} == {p.stem for p in label_paths}, split
    classes = Counter()
    lengths = Counter()
    outside = 0
    for path in label_paths:
        boxes = np.loadtxt(path, ndmin=2)
        assert boxes.shape[1] == 5 and np.isfinite(boxes).all(), path
        assert np.all(boxes[:, 0] == boxes[:, 0].astype(int)), path
        assert np.all((boxes[:, 0] >= 0) & (boxes[:, 0] <= 9)), path
        assert np.all(boxes[:, 3:] > 0), path
        classes.update(boxes[:, 0].astype(int).tolist())
        lengths[len(boxes)] += 1
        corners_low = boxes[:, 1:3] - boxes[:, 3:5] / 2
        corners_high = boxes[:, 1:3] + boxes[:, 3:5] / 2
        outside += int(np.any((corners_low < -1e-5) | (corners_high > 1 + 1e-5), axis=1).sum())
    class_counts[split] = dict(sorted(classes.items()))
    length_counts[split] = dict(sorted(lengths.items()))
    statistics.append({'split': split, 'images': len(image_paths), 'digits': sum(classes.values()),
                       'mean_digits': sum(classes.values()) / len(image_paths), 'boxes_outside_image': outside})
stats = pd.DataFrame(statistics)
stats.to_csv(OUTPUT / 'dataset_statistics.csv', index=False)
pd.DataFrame(class_counts).sort_index().to_csv(OUTPUT / 'class_distribution.csv', index_label='digit')
(OUTPUT / 'dataset_audit.json').write_text(json.dumps({'statistics': statistics, 'class_counts': class_counts,
                                                     'length_counts': length_counts}, indent=2))
print(stats.to_string(index=False))
print('Class counts', class_counts)
print('Sequence lengths', length_counts)

plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
fig, ax = plt.subplots(figsize=(6.7, 2.8))
x = np.arange(10)
for index, split in enumerate(['train', 'val', 'test']):
    counts = np.array([class_counts[split].get(d, 0) for d in range(10)])
    ax.bar(x + (index - 1) * .25, 100 * counts / counts.sum(), .25, label=split)
ax.set(xticks=x, xlabel='Digit class', ylabel='Share of annotated digits (%)')
ax.legend(ncol=3, frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT / 'class_distribution.pdf', bbox_inches='tight')
plt.close(fig)

selected = [0, 1, 3, 4, 5, 50, 82, 125]
fig, axes = plt.subplots(2, 4, figsize=(10, 4.1))
for ax, index in zip(axes.flat, selected):
    stem = f'val_{index:06d}'
    image = Image.open(DATA / 'images/val' / (stem + '.jpg'))
    width, height = image.size
    boxes = np.loadtxt(DATA / 'labels/val' / (stem + '.txt'), ndmin=2)
    ax.imshow(image)
    for digit, cx, cy, w, h in boxes:
        left, top = (cx - w / 2) * width, (cy - h / 2) * height
        ax.add_patch(Rectangle((left, top), w * width, h * height, fill=False, color='#00ff80', linewidth=1.2))
    text = ''.join(str(int(row[0])) for row in sorted(boxes, key=lambda row: row[1]))
    ax.set_title(f'{stem} | GT {text}', fontsize=10)
    ax.axis('off')
fig.tight_layout()
fig.savefig(OUTPUT / 'dataset_examples.png', dpi=220, bbox_inches='tight')
plt.close(fig)
